<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="40%"></a>
</p>

<span style="float:right">
<strong>Credits:</strong>
This notebook adapts material from the <a href="https://docs.ros.org/en/jazzy/">ROS 2 documentation</a> (<a href="https://creativecommons.org/licenses/by/4.0/">CC BY 4.0</a>) and from the Duckietown <em>ROS Basics</em> and <em>ROS 2 Middleware</em> learning experiences.
</span>

# The ROS 2 Graph and Software Layers

Robots run software that performs many tasks at the same time. A Duckiebot acquires camera images, counts the ticks of its wheel encoders, estimates where it is, decides where to go and drives its motors, all concurrently. While it is possible to combine all of that work in one monolithic program, separating it into focused components that exchange data makes each part easier to inspect, test and change.

[Robot Operating System 2 (ROS 2)](https://docs.ros.org/en/jazzy/) provides reusable software infrastructure for robotics applications: communication libraries, a build system and tools for inspecting running systems. These facilities allow application components to exchange data without implementing their own communication mechanisms.

This first conceptual notebook explains the vocabulary used in the rest of the learning experience: the ROS 2 **graph**, its **interfaces**, and which responsibilities belong to the application, which to ROS 2 and which to the operating system. No ROS 2 commands need to be run yet.

## The ROS 2 graph

A **node** is a named software component participating in a ROS 2 application. Giving each node a focused responsibility makes its behaviour easier to inspect, test and change. A node is a piece of software with well-defined inputs and outputs, a clear purpose, and a name that identifies it in the ROS 2 graph. Nodes can be written in different programming languages, such as Python (`rclpy`) or C++ (`rclcpp`).

Consider the Duckiebot driving in a lane. Those responsibilities are separated into nodes of the Duckiebot's ROS 2 stack:

- `camera_node` produces images;
- a lane-following node (or, in this learning experience, *your* node) uses the images to decide how fast to go and how much to turn;
- `kinematics_node` converts that decision, a forward speed `v` and an angular speed `omega`, into a speed for each wheel;
- `wheels_driver_node` sends the wheel speeds to the motors;
- `left_wheel_encoder_driver_node` and `right_wheel_encoder_driver_node` count how much each wheel actually turned.

Nodes exchange data in **messages**. A message has a defined **type**, which specifies its fields. For a stream of data, a node **publishes** messages on a named **topic**, and interested nodes **subscribe** to that topic.

In other words:

- A *topic* is like a pipe where data flows. It has a name and a type, and nodes can publish to it or subscribe to it.
- A *message* is the actual data being sent through that pipe. Its structure is defined by its type.
- *Publishing* means sending data into the topic, *subscribing* means listening for data coming out of it. Multiple nodes can publish to the same topic, and multiple nodes can subscribe to it. The publisher does not need to know who is listening.

These connections form the ROS 2 graph. The simplified data flow of the Duckiebot appears in Figure 1 (`ROBOT` stands for the name of your robot).

<figure style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    camera_node
      |
      | publishes sensor_msgs/CompressedImage
      v
    /ROBOT/camera_node/image/compressed
      |
      | subscribed to by
      v
    your node (e.g. lane following)
      |
      | publishes duckietown_msgs/Twist2DStamped  (v, omega)
      v
    /ROBOT/car_cmd_switch_node/cmd
      |
      v
    kinematics_node
      |
      | publishes duckietown_msgs/WheelsCmdStamped  (vel_left, vel_right)
      v
    /ROBOT/wheels_driver_node/wheels_cmd
      |
      v
    wheels_driver_node  --->  motors
                                 |
    left/right_wheel_encoder_driver_node  <---  encoders
      |
      | publishes duckietown_msgs/WheelEncoderStamped
      v
    /ROBOT/left_wheel_encoder_driver_node/tick
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em;">
    Figure 1: A simplified ROS 2 graph of the Duckiebot: from the camera to the motors and back from the encoders.
  </figcaption>
</figure>

Your node is both a subscriber and a publisher: it receives images and produces car commands. The `kinematics_node` consumes those commands without needing to know how they were computed. This separation allows the algorithm to change while preserving the interface the rest of the robot expects. That agreement includes the topic, the message type and the meaning of the data, such as the units (`v` in m/s, `omega` in rad/s) and sign conventions (positive `omega` turns left).

### Trace the dependency

Suppose the camera continues publishing images, but your node stops producing commands. Which input would the `kinematics_node` lose? Would the presence of image messages alone establish that the whole data path is working?

<details>
<summary>Compare the reasoning</summary>

The `kinematics_node` would lose new car commands, so the wheels stop. Image messages show that data is available at the beginning of this path, but the output of your node would also need inspection to establish whether commands reach the kinematics node. In notebooks 07, 08 and 10 you will use `ros2 topic echo` and `ros2 topic hz` to check each link of such a chain.

</details>

## Beyond streams: services, actions and parameters

Topics are the right interface for a stream of data. Other needs call for other interfaces:

| Need | ROS 2 interface | What the interface provides | Example in this learning experience |
| --- | --- | --- | --- |
| Receive successive images | Topic | a stream of messages | `camera_node/image/compressed` |
| Ask a node to do something once and report back | Service | a request and a response | `drive_node/start`, `drive_node/stop` (`std_srvs/srv/Trigger`) |
| Navigate to a goal while reporting progress | Action | a goal, feedback, a final result and cancellation | not used here |
| Configure a node's behaviour | Parameter | a named configuration value owned by the node | `v`, `omega` of `drive_node` |

A **parameter** is not a Python constant: the node must declare it, and then anybody on the graph can read or change it (`ros2 param get/set`, or a slider in these notebooks) while the node keeps running. You will meet all of these interfaces in command output in the next notebooks.

## Software layers

The nodes contain the application logic: reading a camera, estimating a pose, or calculating a command. ROS 2 supplies common communication facilities so each application does not have to implement them from scratch.

A Python node uses the ROS 2 client library, `rclpy`, to create publishers and subscriptions. Beneath the client library, the ROS Middleware (RMW) interface connects ROS 2 to a middleware implementation. That implementation handles communication and **discovery**, the process through which participants find one another. On the Duckiebot the middleware is **zenoh** (`rmw_zenoh_cpp`), and discovery goes through a *zenoh router* running on the robot; notebook 04 is about it.

The relationship between these layers appears in Figure 2.

<figure style="margin:1.5em auto; text-align:center;">
  <pre style="display:inline-block; margin:0; text-align:left;">
    application logic (what your node is meant to do)
      |
    Python client library: rclpy
      |
    common ROS client-library layer: rcl
      |
    ROS Middleware (RMW) interface
      |
    middleware implementation: rmw_zenoh_cpp (zenoh)
      |
    Linux: processes, Docker containers, the network
  </pre>
  <figcaption style="font-size:0.9em; margin-top:0.6em;">
    Figure 2: A simplified path from Python application code to the communication middleware.
  </figcaption>
</figure>

Despite its name, ROS 2 does not replace the operating system. Here it runs on Linux, which provides process scheduling, device access and networking; on the Duckiebot every node runs inside a Docker container. ROS 2 builds on those facilities to connect the application nodes.

For example, the publisher code chooses the message contents and when to publish. It calls `rclpy` to publish the message; the communication layers handle its exchange with compatible subscribers, whether they run in the same process, in another container, or on your laptop.

## ROS 2 is distributed

The graph describes **communication relationships**, not where programs run. Several nodes can share one process, run in separate containers on the robot, or communicate across computers. In this learning experience:

- the driver nodes (`camera_node`, `wheels_driver_node`, the encoder nodes, `led_driver_node`, `tof_driver_node`) and the car interface (`kinematics_node`, `car_cmd_switch_node`, `velocity_to_pose_node`) run on the robot, each in its own container, started with `dts stack up`;
- your nodes run either on the robot (`dts code workbench`) or inside this editor, on your laptop;
- these notebooks themselves are a node (the `RosSession` cell) that subscribes to the robot's topics and calls its services.

Moving your node from the laptop to the robot would change performance metrics such as latency, but not its role: it would still subscribe to images and publish commands. Communication only requires that every participant reaches the same zenoh router over the network (the robot's nodes as peers, everything outside the robot as clients of that router; notebook 04).

The `ros2` command-line tools inspect the graph visible from a terminal and check whether messages actually arrive. A ROS 2 graph connects focused nodes through shared interfaces; following its messages identifies which component produces the data another component needs.

## Further reading

The ROS 2 Jazzy documentation introduces [ROS 2 concepts](https://docs.ros.org/en/jazzy/Concepts.html), explains [topics](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Topics.html), [services](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Services.html), [actions](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Actions.html) and [parameters](https://docs.ros.org/en/jazzy/Concepts/Basic/About-Parameters.html), and describes the available [middleware implementations](https://docs.ros.org/en/jazzy/Concepts/Advanced/About-Middleware-Implementations.html).

For additional insight into logical and physical architectures, see the Duckietown lecture on [System Architectures](https://hubs.ly/Q02b8kCV0).

Let's see these concepts in command output in [the next notebook about nodes, topics and interfaces](../notebooks/03_nodes_topics_interfaces.ipynb).